# Lab 09: Attention and a Tiny Transformer

            **Duration:** 3 hours  
            **Lecture alignment:** Week 9 — Attention mechanisms and Transformers  
            **CLO mapping:** CLO-1, CLO-2, CLO-3  
            **Framework:** PyTorch (standalone, credential-free, CPU smoke-test with optional GPU)

            ## Learning objectives

            - Implement scaled dot-product attention and causal masking.
- Use positional representations and multi-head self-attention in a tiny Transformer.
- Interpret attention weights cautiously and distinguish common Transformer objectives.

            ## Three-hour activity plan

            - 0–35 min: attention calculation and unit tests
- 35–65 min: masking and positional encoding
- 65–120 min: Tiny Transformer implementation/training
- 120–155 min: attention visualization and errors
- 155–180 min: BERT/GPT/T5 objective comparison and checks


## Book grounding

            - Zhang, Lipton, Li, and Smola, *Dive into Deep Learning*, Cambridge University Press, 2024.
- Prince, *Understanding Deep Learning*, MIT Press, 2023.
- Bishop and Bishop, *Deep Learning: Foundations and Concepts*, Springer, 2024.

            The notebook paraphrases concepts and supplies original code; it does not reproduce book text.


In [ ]:
from pathlib import Path
import json, math, os, random, time
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset

FAST_MODE = True
RUN_EXTENSION = False
SEED = 20269
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(min(2, os.cpu_count() or 1))

if Path("/content").exists():
    ARTIFACT_DIR = Path("/content/artifacts/lab_09")
else:
    ARTIFACT_DIR = Path.cwd() / "tmp" / "course_build" / "runtime" / "lab_09"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print({"lab": 9, "device": str(DEVICE), "fast_mode": FAST_MODE,
       "artifacts": str(ARTIFACT_DIR), "torch": torch.__version__})


## Predict before running

Will the first query position learn to attend strongly to the last key position? Explain why a visible weight would still not prove causal importance.

Record a brief prediction in your own words before executing the experiment, then revisit it in the exit reflection.


## Activity 1 — Scaled dot-product attention, masks, and positional information


In [ ]:
def scaled_dot_attention(q,k,v,mask=None):
    scores=q@k.transpose(-2,-1)/math.sqrt(q.shape[-1])
    if mask is not None: scores=scores.masked_fill(mask,float("-inf"))
    weights=torch.softmax(scores,dim=-1)
    return weights@v,weights
q=torch.randn(2,6,8);k=torch.randn(2,6,8);v=torch.randn(2,6,8)
causal=torch.triu(torch.ones(6,6,dtype=torch.bool),diagonal=1)
attended,manual_weights=scaled_dot_attention(q,k,v,causal)
positions=torch.arange(8).unsqueeze(1);dimensions=torch.arange(0,16,2)
pe=torch.zeros(8,16);rates=torch.exp(-math.log(10000.0)*dimensions/16)
pe[:,0::2]=torch.sin(positions*rates);pe[:,1::2]=torch.cos(positions*rates)
assert torch.allclose(manual_weights.sum(-1),torch.ones(2,6),atol=1e-5)
print({"attention":tuple(attended.shape),"causal_masked_weight_max":manual_weights[:,causal].max().item()})


## Activity 2 — Tiny Transformer for a sequence-relation task


In [ ]:
def make_relation_sequences(n,length=8,vocab=10):
    seq=torch.randint(1,vocab+1,(n,length));labels=torch.randint(0,2,(n,))
    for i,label in enumerate(labels.tolist()):
        if label==1:seq[i,-1]=seq[i,0]
        else:seq[i,-1]=(seq[i,0]%vocab)+1
    order=torch.randperm(n);return seq[order],labels[order]
seq,labels=make_relation_sequences(900 if FAST_MODE else 4000)
split=int(.8*len(seq));seqtr,seqte,ytr,yte=seq[:split],seq[split:],labels[:split],labels[split:]

class TinyTransformer(nn.Module):
    def __init__(self,vocab=10,d_model=24,heads=3,max_len=8):
        super().__init__();self.embedding=nn.Embedding(vocab+1,d_model);self.position=nn.Parameter(torch.randn(1,max_len,d_model)*.02)
        self.attn=nn.MultiheadAttention(d_model,heads,batch_first=True)
        self.norm1=nn.LayerNorm(d_model);self.ff=nn.Sequential(nn.Linear(d_model,48),nn.GELU(),nn.Linear(48,d_model));self.norm2=nn.LayerNorm(d_model)
        self.head=nn.Linear(2*d_model,2)
    def forward(self,tokens,return_attention=False):
        x=self.embedding(tokens)+self.position[:,:tokens.shape[1]]
        a,w=self.attn(x,x,x,need_weights=True,average_attn_weights=False);x=self.norm1(x+a);x=self.norm2(x+self.ff(x))
        logits=self.head(torch.cat([x[:,0],x[:,-1]],dim=1))
        return (logits,w) if return_attention else logits
model=TinyTransformer().to(DEVICE);opt=torch.optim.AdamW(model.parameters(),lr=.01,weight_decay=1e-3)
loader=DataLoader(TensorDataset(seqtr,ytr),batch_size=64,shuffle=True,generator=torch.Generator().manual_seed(SEED));history=[]
for _ in range(12 if FAST_MODE else 35):
    model.train();total=0
    for xb,yb in loader:
        xb,yb=xb.to(DEVICE),yb.to(DEVICE);opt.zero_grad();loss=F.cross_entropy(model(xb),yb);loss.backward();opt.step();total+=loss.item()*len(xb)
    history.append(total/len(seqtr))
model.eval()
with torch.no_grad():logits,attn_weights=model(seqte.to(DEVICE),True);acc=(logits.argmax(1).cpu()==yte).float().mean().item()
print({"test_accuracy":acc,"attention_shape":tuple(attn_weights.shape)})


## Activity 3 — Inspect a head, then distinguish BERT-, GPT-, and T5-style objectives


In [ ]:
fig,axes=plt.subplots(1,2,figsize=(10,3.8));axes[0].plot(history);axes[0].set(title="Transformer training loss",xlabel="epoch")
im=axes[1].imshow(attn_weights[0,0].cpu(),cmap="magma",vmin=0,vmax=1);axes[1].set(title="Example: head 0",xlabel="key position",ylabel="query position");fig.colorbar(im,ax=axes[1])
fig.tight_layout();fig.savefig(ARTIFACT_DIR/"attention_transformer.png",dpi=150);plt.show()
objective_table={"BERT-style":"masked-token encoder","GPT-style":"causal next-token decoder","T5-style":"text-to-text encoder-decoder"}
print(json.dumps(objective_table,indent=2));torch.save(model.state_dict(),ARTIFACT_DIR/"tiny_transformer.pt")


## Automated checks


In [ ]:
assert attended.shape==q.shape and manual_weights[:,causal].max()<1e-6
assert attn_weights.shape[1]==3 and torch.allclose(attn_weights.sum(-1),torch.ones_like(attn_weights.sum(-1)),atol=1e-4)
assert history[-1]<history[0] and acc>.70
assert (ARTIFACT_DIR/"attention_transformer.png").exists()
print("All Lab 09 checks passed.")


## Deliverables

                - Attention/mask unit tests
- Trained tiny Transformer checkpoint
- Loss/attention figure
- BERT-, GPT-, and T5-style objective comparison

                Submit the executed notebook and the files created in `/content/artifacts/lab_09/`.


## Disabled extension

The following challenge is intentionally disabled by default so the CPU baseline stays quick.


In [ ]:
if RUN_EXTENSION:
    print("Extension: use the causal mask to train a next-token model on generated arithmetic sequences.")
else:
    print("Extension disabled: causal next-token modeling and length extrapolation.")


## Exit reflection

In 4–6 sentences, state: (1) whether your prediction was supported, (2) the strongest evidence,
(3) one failure mode or limitation, and (4) the next experiment you would run. Include at least
one measured value rather than only a general claim.
